In [33]:
import pandas as pd
df = pd.read_csv("C:\\Users\\Acer\\Downloads\\COMP4501-FYP\\cleaned_dataset_final\\Sunscreen_updated_cleaned.csv")


In [34]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)   # allow pandas to use full width

In [35]:
import pandas as pd
import re

# -------------------------------------------------------------------
# 1. Clean seller_chat_time_reply to numeric hours
# -------------------------------------------------------------------

def clean_chat_time(val):
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if s == 'hitungan jam':
        return 1.0
    if s == 'hitungan menit':
        return 0.5
    match = re.search(r'([\d,\.]+)\s*(jam|menit|hours?|hrs?)', s)
    if match:
        num_str = match.group(1).replace(',', '.')
        try:
            num = float(num_str)
        except:
            return None
        unit = match.group(2)
        if unit.startswith('jam') or unit.startswith('hour') or unit.startswith('hr'):
            return num
        elif unit.startswith('menit'):
            return num / 60.0
        else:
            return None
    try:
        return float(s)
    except:
        return None

df['seller_chat_time_reply'] = df['seller_chat_time_reply'].apply(clean_chat_time)

# -------------------------------------------------------------------
# 2. Clean seller_joined_date to numeric years (relative to 2026-03-21)
# -------------------------------------------------------------------

REF_DATE = pd.to_datetime('2026-03-21')

def parse_absolute(s):
    s_clean = re.sub(r'\s*GMT.*$', '', s)
    s_clean = re.sub(r'\s*\(.*$', '', s_clean)
    try:
        return pd.to_datetime(s_clean, errors='coerce')
    except:
        return pd.NaT

def relative_to_years(s):
    match = re.match(r'(\d+)\s+(tahun|bulan|hari)\s+lalu', s)
    if not match:
        return None
    num = int(match.group(1))
    unit = match.group(2)
    if unit == 'tahun':
        return float(num)
    elif unit == 'bulan':
        return num / 12.0
    else:  # hari
        return num / 365.25

def convert_seller_joined_date(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    if 'lalu' in val_str:
        years = relative_to_years(val_str)
        if years is not None:
            return years
    dt = parse_absolute(val_str)
    if pd.notna(dt):
        return (REF_DATE - dt).days / 365.25
    return None

df['seller_joined_date'] = df['seller_joined_date'].apply(convert_seller_joined_date)

### Clean "Dikirim Dari"

In [36]:
region_map = {
    # Jabodetabek (or separate Jakarta)
    'KOTA JAKARTA SELATAN': 'Jabodetabek',
    'KOTA JAKARTA BARAT': 'Jabodetabek',
    'KOTA JAKARTA UTARA': 'Jabodetabek',
    'KOTA JAKARTA TIMUR': 'Jabodetabek',
    'KOTA JAKARTA PUSAT': 'Jabodetabek',
    'KOTA BEKASI': 'Jabodetabek',
    'KOTA DEPOK': 'Jabodetabek',
    'KOTA TANGERANG': 'Jabodetabek',
    'KOTA TANGERANG SELATAN': 'Jabodetabek',
    'KAB. BEKASI': 'Jabodetabek',
    'KAB. TANGERANG': 'Jabodetabek',
    'KAB. BOGOR': 'Jabodetabek',
    
    # Java (non-Jabodetabek) – add more as needed
    'KOTA BANDUNG': 'Java',
    'KAB. BANDUNG': 'Java',
    'KAB. BANDUNG BARAT': 'Java',
    'KOTA CIREBON': 'Java',
    'KOTA TASIKMALAYA': 'Java',
    'KAB. TASIKMALAYA': 'Java',
    'KOTA SUKABUMI': 'Java',
    'KAB. CIANJUR': 'Java',
    'KAB. GARUT': 'Java',
    'KAB. MAJALENGKA': 'Java',
    'KAB. PURBALINGGA': 'Java',
    'KAB. CILACAP': 'Java',
    'KAB. BANYUMAS': 'Java',
    'KAB. PEMALANG': 'Java',
    'KAB. PEKALONGAN': 'Java',
    'KOTA PEKALONGAN': 'Java',
    'KAB. BATANG': 'Java',
    'KAB. KENDAL': 'Java',
    'KOTA SEMARANG': 'Java',
    'KAB. DEMAK': 'Java',
    'KAB. REMBANG': 'Java',
    'KAB. BLORA': 'Java',
    'KAB. PATI': 'Java',
    'KAB. JEPARA': 'Java',
    'KAB. KUDUS': 'Java',
    'KOTA SURAKARTA (SOLO)': 'Java',
    'KAB. SUKOHARJO': 'Java',
    'KAB. KARANGANYAR': 'Java',
    'KAB. WONOGIRI': 'Java',
    'KOTA YOGYAKARTA': 'Java',
    'KAB. SLEMAN': 'Java',
    'KAB. BANTUL': 'Java',
    'KAB. GUNUNG KIDUL': 'Java',
    'KAB. KULON PROGO': 'Java',
    'KOTA MALANG': 'Java',
    'KAB. MALANG': 'Java',
    'KAB. PASURUAN': 'Java',
    'KOTA PASURUAN': 'Java',
    'KOTA PROBOLINGGO': 'Java',  # if appears
    'KAB. PROBOLINGGO': 'Java',
    'KOTA SURABAYA': 'Java',
    'KAB. SIDOARJO': 'Java',
    'KAB. GRESIK': 'Java',
    'KAB. LAMONGAN': 'Java',
    'KAB. BOJONEGORO': 'Java',
    'KAB. TUBAN': 'Java',
    'KAB. NGAWI': 'Java',
    'KAB. MADIUN': 'Java',
    'KOTA MADIUN': 'Java',
    'KAB. MAGETAN': 'Java',
    'KAB. PONOROGO': 'Java',
    'KAB. PACITAN': 'Java',
    'KAB. TRENGGALEK': 'Java',
    'KOTA BLITAR': 'Java',
    'KAB. BLITAR': 'Java',
    'KAB. KEDIRI': 'Java',
    'KOTA KEDIRI': 'Java',
    'KAB. NGANJUK': 'Java',
    'KAB. JOMBANG': 'Java',
    'KAB. MOJOKERTO': 'Java',
    'KOTA MOJOKERTO': 'Java',
    'KAB. BANGKALAN': 'Java',   # Madura (often grouped with East Java)
    'KAB. PAMEKASAN': 'Java',
    'KAB. SAMPANG': 'Java',
    'KAB. SUMENEP': 'Java',
    'KAB. SERANG': 'Java',       # Banten
    'KAB. PANDEGLANG': 'Java',
    'KAB. LEBAK': 'Java',
    'KOTA CILEGON': 'Java',
    
    # Sumatra
    'KOTA MEDAN': 'Sumatra',
    'KOTA PEKANBARU': 'Sumatra',
    'KOTA PALEMBANG': 'Sumatra',
    'KOTA PADANG': 'Sumatra',
    'KOTA BANDAR LAMPUNG': 'Sumatra',
    'KOTA PANGKAL PINANG': 'Sumatra',
    'KOTA TANJUNG PINANG': 'Sumatra',
    'KOTA BATAM': 'Sumatra',
    'KAB. KARIMUN': 'Sumatra',
    'KAB. BANGKA': 'Sumatra',
    # ... add more as needed
    
    # Kalimantan
    'KOTA BANJARMASIN': 'Kalimantan',
    'KOTA BANJARBARU': 'Kalimantan',
    'KAB. BANJAR': 'Kalimantan',
    'KAB. HULU SUNGAI TENGAH': 'Kalimantan',
    'KAB. HULU SUNGAI UTARA': 'Kalimantan',
    'KOTA BALIKPAPAN': 'Kalimantan',
    'KOTA SAMARINDA': 'Kalimantan',
    # ... add more
    
    # Sulawesi
    'KOTA MAKASSAR': 'Sulawesi',
    'KAB. GOWA': 'Sulawesi',
    'KAB. SOPPENG': 'Sulawesi',
    'KAB. SIDENRENG RAPPANG/RAPANG': 'Sulawesi',
    'KOTA MANADO': 'Sulawesi',
    # ... add more
    
    # Bali & Nusa Tenggara
    'KOTA DENPASAR': 'Bali & Nusa Tenggara',
    
    # Maluku & Papua
    # (none in your array, but could add)
    
    # Overseas
    'Luar Negeri': 'Overseas',
    
    # Unknown
    None: 'Unknown'
}

# Apply mapping
df['Dikirim Dari'] = df['Dikirim Dari'].map(region_map).fillna('Unknown')
df['Dikirim Dari'].value_counts()

Dikirim Dari
Java                    2279
Jabodetabek             1624
Unknown                 1354
Sumatra                  431
Kalimantan               191
Sulawesi                 177
Bali & Nusa Tenggara      32
Name: count, dtype: int64

### Drop Breadcrumb

In [37]:
df = df.drop(columns=[col for col in df.columns if "breadcrumb" in col.lower()])

In [38]:
df.head()

,id,title,rating,reviews,initial_price,final_price,stock,favorite,seller_name,seller_rating,seller_products,seller_chats_responded_percentage,seller_chat_time_reply,seller_joined_date,seller_followers,sold,domain,brand,category_id,flash_sale,product_variation,gmv_cal,category_url,vouchers,is_available,seller_id,product_ratings,Masa Penyimpanan,Stok,Dikirim Dari,Berat Produk,Formulasi,"No. Izin Edar (BPOM, lainnya)",Tanggal Kedaluwarsa,SPF,Negara Asal,Volume,Jenis Kulit,Jumlah Produk Dalam Kemasan,Ukuran Per Produk,variations_count,voucher_status,image_count,video_count,review,discount
0,74170424,Skin Aqua UV Moisture Milk SPF 50 | Moisture Gel SPF 30 | Mild Milk SPF 25 | Whitening Milk SPF 50 | Tone Up UV Essence Lavender | Mint Green SPF 50+ | Sunscreen Mist,4.92,16038,49900.0,41300.0,1580,10,megabeautyfashindo,4.91,784,99,1.0,7.000000,114400,10,NaN,NaN,NaN,NaN,NaN,413000.0,NaN,NaN,NaN,NaN,NaN,24 Bulan,1580.0,Jabodetabek,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,0,8,0,0,8600.0
1,19980336891,Skin Aqua UV Whitening Milk SPF 50 PA++++ Sunscreen,4.78,112,61000.0,42500.0,50,128,DesakiCosmetik,4.87,39,92,1.0,0.333333,4200,468,NaN,NaN,NaN,NaN,NaN,19890000.0,NaN,NaN,NaN,NaN,NaN,12 Bulan,50.0,Jabodetabek,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,6,0,0,18500.0
2,18865050074,AZARINE BT21 BTS Body Guard Moisturizer Sunscreen Serum SPF 50 PA++++ | Magical Luv Sweet Treats Baby Sunproof Monster Sun O Clock Young and Free,5.00,18,65000.0,62200.0,16,5,Made By Caramel,4.98,414,100,1.0,5.000000,4400,40,NaN,NaN,NaN,NaN,NaN,2488000.0,NaN,NaN,NaN,NaN,NaN,NaN,16.0,Sumatra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,0,8,0,0,2800.0
3,6862950349,KUMARA - SUNSCREEN Wardah | Madame Gie Madame Protect Me Sunscreen SPF 30 - SkinCare,4.79,567,31000.0,26500.0,402,229,KUMARA STORE KOSMETIK,4.79,741,92,1.0,4.000000,61100,2700,NaN,NaN,NaN,NaN,NaN,71550000.0,NaN,NaN,NaN,NaN,NaN,24 Bulan,402.0,Java,150g,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,0,6,0,0,4500.0
4,22877274329,Madame Gie Madame Protect Me Sunscreen Spray Mist SPF 50 PA++++ | Isi 50ml & 100ml,4.86,35,34000.0,29000.0,9,50,cherrishopie,4.82,460,81,1.0,8.000000,9600,138,NaN,NaN,NaN,NaN,NaN,4002000.0,NaN,NaN,NaN,NaN,NaN,24 Bulan,9.0,Unknown,NaN,Spray,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0,7,0,0,5000.0


In [39]:
def clean_flashsale_and_drop(df):
    """
    - Drop unnecessary columns
    - Convert flash_sale to binary (1/0)
    """

    # ---------- 1. Drop columns ---------- #
    cols_to_drop = ["domain", "category_id", "product_variation", "category_url", "vouchers", "is_available", "seller_id", "product_ratings"]

    df = df.drop(
        columns=[col for col in cols_to_drop if col in df.columns],
        errors="ignore"
    )

    # ---------- 2. Convert flash_sale ---------- #
    if "flash_sale" in df.columns:
        df["flash_sale"] = df["flash_sale"].map({
            True: 1,
            False: 0,
            "True": 1,
            "False": 0
        })

        # handle unexpected / missing values
        df["flash_sale"] = df["flash_sale"].fillna(0).astype(int)

    return df

df = clean_flashsale_and_drop(df)

In [40]:
df.head()

,id,title,rating,reviews,initial_price,final_price,stock,favorite,seller_name,seller_rating,seller_products,seller_chats_responded_percentage,seller_chat_time_reply,seller_joined_date,seller_followers,sold,brand,flash_sale,gmv_cal,Masa Penyimpanan,Stok,Dikirim Dari,Berat Produk,Formulasi,"No. Izin Edar (BPOM, lainnya)",Tanggal Kedaluwarsa,SPF,Negara Asal,Volume,Jenis Kulit,Jumlah Produk Dalam Kemasan,Ukuran Per Produk,variations_count,voucher_status,image_count,video_count,review,discount
0,74170424,Skin Aqua UV Moisture Milk SPF 50 | Moisture Gel SPF 30 | Mild Milk SPF 25 | Whitening Milk SPF 50 | Tone Up UV Essence Lavender | Mint Green SPF 50+ | Sunscreen Mist,4.92,16038,49900.0,41300.0,1580,10,megabeautyfashindo,4.91,784,99,1.0,7.000000,114400,10,NaN,0,413000.0,24 Bulan,1580.0,Jabodetabek,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,0,8,0,0,8600.0
1,19980336891,Skin Aqua UV Whitening Milk SPF 50 PA++++ Sunscreen,4.78,112,61000.0,42500.0,50,128,DesakiCosmetik,4.87,39,92,1.0,0.333333,4200,468,NaN,0,19890000.0,12 Bulan,50.0,Jabodetabek,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,6,0,0,18500.0
2,18865050074,AZARINE BT21 BTS Body Guard Moisturizer Sunscreen Serum SPF 50 PA++++ | Magical Luv Sweet Treats Baby Sunproof Monster Sun O Clock Young and Free,5.00,18,65000.0,62200.0,16,5,Made By Caramel,4.98,414,100,1.0,5.000000,4400,40,NaN,0,2488000.0,NaN,16.0,Sumatra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,0,8,0,0,2800.0
3,6862950349,KUMARA - SUNSCREEN Wardah | Madame Gie Madame Protect Me Sunscreen SPF 30 - SkinCare,4.79,567,31000.0,26500.0,402,229,KUMARA STORE KOSMETIK,4.79,741,92,1.0,4.000000,61100,2700,NaN,0,71550000.0,24 Bulan,402.0,Java,150g,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,0,6,0,0,4500.0
4,22877274329,Madame Gie Madame Protect Me Sunscreen Spray Mist SPF 50 PA++++ | Isi 50ml & 100ml,4.86,35,34000.0,29000.0,9,50,cherrishopie,4.82,460,81,1.0,8.000000,9600,138,NaN,0,4002000.0,24 Bulan,9.0,Unknown,NaN,Spray,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0,7,0,0,5000.0


### Clean Masa Penyimpanan

In [41]:
df['Masa Penyimpanan'].value_counts()

Masa Penyimpanan
24 Bulan             3589
12 Bulan             1198
6 Bulan               132
36 Bulan               99
36 Months              72
36 bulan               46
3 Tahun                20
simpan suhu ruang      14
36                     11
30 Bulan               10
3 Bulan                 7
3 tahun                 6
48 bulan                6
48 BULAN                6
2 Bulan                 5
5 tahun                 4
3 th                    4
48 Bulan                4
30 bulan                4
1 Bulan                 4
4 tahun                 4
36Bulan                 3
11 Juni 2026            2
2025                    2
3tahun                  2
34 Bulan                2
EXP 2026                2
36 BULAN                2
2023                    2
Juni 2026               2
Mei 2026                2
33 Bulan                2
5 Tahun                 2
36 bulAN                2
EXP 2025                2
-                       2
Sampai masa exsp        2
3th                  

In [42]:
import pandas as pd
import re

def convert_to_months(x):
    if pd.isna(x):
        return None
    
    x = str(x).lower().strip()

    # ---------- HANDLE NOISE ---------- #
    if any(word in x for word in ["simpan", "exp", "-", "sampai"]):
        return None

    # ---------- YEARS → MONTHS ---------- #
    if any(word in x for word in ["tahun", "th", "thn", "y"]):
        num = re.findall(r"\d+", x)
        if num:
            return int(num[0]) * 12

    # ---------- MONTHS ---------- #
    if "bulan" in x:
        num = re.findall(r"\d+", x)
        if num:
            return int(num[0])

    # ---------- NUMBER ONLY ---------- #
    if re.fullmatch(r"\d+", x):
        return int(x)

    return None

df["storage_duration_months"] = df["Masa Penyimpanan"].apply(convert_to_months)

### Clean

In [43]:
df['Berat Produk'].value_counts()

Berat Produk
50g       246
30g       167
100g      148
60g        55
40g        40
20g        35
10g        35
200g       28
150g       26
250g       23
300g       18
500g       14
0.052g     10
75g         8
15g         7
0.075g      6
80g         6
55g         4
71g         2
100.0g      2
128g        2
60.0g       2
115g        2
7g          2
120g        2
35.0g       1
1.25g       1
275g        1
308g        1
30.0g       1
89g         1
97g         1
0.14g       1
72g         1
50.0g       1
22g         1
0.047g      1
12g         1
Name: count, dtype: int64

In [44]:
import pandas as pd
import re

# ---------- 1. Extract numeric weight (grams) ---------- #
def extract_weight(x):
    if pd.isna(x):
        return None
    
    x = str(x).lower().strip()

    # extract numeric value
    num = re.findall(r"\d+\.?\d*", x)

    if num:
        return float(num[0])
    
    return None


# ---------- 2. Group into ranges ---------- #
def group_weight(x):
    if pd.isna(x):
        return "unknown"
    elif x <= 50:
        return "very_light"
    elif x <= 100:
        return "light"
    elif x <= 200:
        return "medium"
    elif x <= 500:
        return "heavy"
    else:
        return "very_heavy"


# ---------- 3. Main function ---------- #
def clean_berat_produk(df):
    """
    - Convert 'Berat Produk' to numeric grams
    - Create weight group
    """

    if "Berat Produk" not in df.columns:
        return df

    # Extract numeric weight
    df["berat_gram"] = df["Berat Produk"].apply(extract_weight)

    # 🔥 Optional: remove suspicious values (<1g)
    df.loc[df["berat_gram"] < 1, "berat_gram"] = None

    # Create grouped category
    df["berat_group"] = df["berat_gram"].apply(group_weight)

    return df

df = clean_berat_produk(df)

In [45]:
df['berat_group'].value_counts()

berat_group
unknown       5203
very_light     538
light          230
medium          60
heavy           57
Name: count, dtype: int64

## Clean Formulasi

In [46]:
df['Formulasi'].value_counts()

Formulasi
Cair                                        372
Krim                                        188
Gel                                         179
Cream                                        60
Spray                                        43
Stik                                         29
gel                                          22
Lotion                                       18
cream                                        15
krim                                         12
Lainnya                                      12
Tabur                                        10
Padat                                        10
GEL                                           6
CREAM                                         4
lotion                                        3
Minyak                                        3
Kombinasi                                     2
Cleanser, Serum, Moisture Gel, Sunscreen      2
Krim Losion                                   2
Milk, gel                     

In [47]:
import pandas as pd
import re

def clean_formulasi(x):
    if pd.isna(x):
        return "others"
    
    x = str(x).lower()

    # normalize
    x = re.sub(r"[^a-z0-9 ]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    # ---------- RULES ---------- #

    # Gel
    if "gel" in x:
        return "gel"

    # Cream
    elif any(word in x for word in ["krim", "cream"]):
        return "cream"

    # Liquid
    elif any(word in x for word in [
        "cair", "lotion", "serum", "milk", "essence"
    ]):
        return "liquid"

    # Spray
    elif "spray" in x:
        return "spray"

    # Solid
    elif any(word in x for word in [
        "stik", "stick", "padat", "tabur", "balm"
    ]):
        return "solid"

    # Others
    else:
        return "others"

df["Formulasi"] = df["Formulasi"].apply(clean_formulasi)

In [48]:
df['Formulasi'].value_counts()

Formulasi
others    5094
liquid     397
cream      282
gel        222
solid       50
spray       43
Name: count, dtype: int64

### Clean SPF

In [49]:
df['SPF'].value_counts()

SPF
50.0    735
30.0    241
35.0     88
45.0     87
40.0     15
15.0      4
25.0      3
33.0      3
46.0      2
0.0       2
60.0      2
20.0      2
43.0      1
90.0      1
Name: count, dtype: int64

In [50]:
import pandas as pd

def group_spf(x):
    if pd.isna(x):
        return "unknown"
    
    try:
        x = float(x)
    except:
        return "unknown"

    if x == 0:
        return "no_spf"
    elif x < 30:
        return "low"
    elif x < 50:
        return "medium"
    elif x < 70:
        return "high"
    else:
        return "very_high"

df["SPF"] = df["SPF"].apply(group_spf)
df['SPF'].value_counts()

SPF
unknown      4902
high          737
medium        437
low             9
no_spf          2
very_high       1
Name: count, dtype: int64

## Clean Negara Asal

In [51]:
df['Negara Asal'].value_counts()

Negara Asal
Indonesia    817
Jepang        23
Korea         19
Thailand      19
China         16
Prancis        8
Lokal          2
Perancis       2
Amerika        1
Lainnya        1
Name: count, dtype: int64

In [52]:
import pandas as pd

def clean_negara_asal(x):
    if pd.isna(x):
        return "others"
    
    x = str(x).lower().strip()

    # Indonesia (including Lokal)
    if "indonesia" in x or "lokal" in x:
        return "indonesia"
    
    # China
    elif "china" in x:
        return "china"
    
    # Korea
    elif "korea" in x:
        return "korea"
    
    # Japan
    elif "jepang" in x:
        return "japan"
    
    # Others
    else:
        return "others"

df["Negara Asal"] = df["Negara Asal"].apply(clean_negara_asal)
df['Negara Asal'].value_counts()

Negara Asal
others       5211
indonesia     819
japan          23
korea          19
china          16
Name: count, dtype: int64

### Clean Volume

In [53]:
df['Volume'].value_counts()

Volume
50ml       206
30ml       139
60ml        84
40ml        69
100ml       63
20ml        22
10ml        20
35ml        14
200ml       10
300ml        9
500ml        8
150ml        6
5ml          5
23ml         4
27ml         4
400ml        4
15ml         4
80ml         2
52ml         2
60.0ml       2
250ml        2
170ml        2
65ml         2
180ml        2
89ml         2
25ml         2
7ml          2
40.0ml       1
30.0ml       1
12.5ml       1
50.0ml       1
260ml        1
22ml         1
100.0ml      1
Name: count, dtype: int64

In [54]:
import pandas as pd
import re

# ---------- 1. Extract numeric volume ---------- #
def extract_volume(x):
    if pd.isna(x):
        return None
    
    x = str(x).lower().strip()

    # extract number
    num = re.findall(r"\d+\.?\d*", x)

    if num:
        return float(num[0])
    
    return None


# ---------- 2. Group into ranges ---------- #
def group_volume(x):
    if pd.isna(x):
        return "unknown"
    elif x <= 20:
        return "very_small"
    elif x <= 50:
        return "small"
    elif x <= 100:
        return "medium"
    elif x <= 300:
        return "large"
    else:
        return "very_large"


# ---------- 3. Main function ---------- #
def clean_volume(df):
    """
    - Convert 'Volume' to numeric ml
    - Create grouped categories
    """

    if "Volume" not in df.columns:
        return df

    # Extract numeric value
    df["volume_ml"] = df["Volume"].apply(extract_volume)

    # Create grouped category
    df["volume_group"] = df["volume_ml"].apply(group_volume)

    return df

df = clean_volume(df)
df['volume_group'].value_counts()

volume_group
unknown       5390
small          442
medium         158
very_small      54
large           32
very_large      12
Name: count, dtype: int64

In [55]:
df.head()

,id,title,rating,reviews,initial_price,final_price,stock,favorite,seller_name,seller_rating,seller_products,seller_chats_responded_percentage,seller_chat_time_reply,seller_joined_date,seller_followers,sold,brand,flash_sale,gmv_cal,Masa Penyimpanan,Stok,Dikirim Dari,Berat Produk,Formulasi,"No. Izin Edar (BPOM, lainnya)",Tanggal Kedaluwarsa,SPF,Negara Asal,Volume,Jenis Kulit,Jumlah Produk Dalam Kemasan,Ukuran Per Produk,variations_count,voucher_status,image_count,video_count,review,discount,storage_duration_months,berat_gram,berat_group,volume_ml,volume_group
0,74170424,Skin Aqua UV Moisture Milk SPF 50 | Moisture Gel SPF 30 | Mild Milk SPF 25 | Whitening Milk SPF 50 | Tone Up UV Essence Lavender | Mint Green SPF 50+ | Sunscreen Mist,4.92,16038,49900.0,41300.0,1580,10,megabeautyfashindo,4.91,784,99,1.0,7.000000,114400,10,NaN,0,413000.0,24 Bulan,1580.0,Jabodetabek,NaN,others,NaN,NaN,unknown,others,NaN,NaN,NaN,NaN,7,0,8,0,0,8600.0,24.0,NaN,unknown,NaN,unknown
1,19980336891,Skin Aqua UV Whitening Milk SPF 50 PA++++ Sunscreen,4.78,112,61000.0,42500.0,50,128,DesakiCosmetik,4.87,39,92,1.0,0.333333,4200,468,NaN,0,19890000.0,12 Bulan,50.0,Jabodetabek,NaN,others,NaN,NaN,unknown,others,NaN,NaN,NaN,NaN,0,0,6,0,0,18500.0,12.0,NaN,unknown,NaN,unknown
2,18865050074,AZARINE BT21 BTS Body Guard Moisturizer Sunscreen Serum SPF 50 PA++++ | Magical Luv Sweet Treats Baby Sunproof Monster Sun O Clock Young and Free,5.00,18,65000.0,62200.0,16,5,Made By Caramel,4.98,414,100,1.0,5.000000,4400,40,NaN,0,2488000.0,NaN,16.0,Sumatra,NaN,others,NaN,NaN,unknown,others,NaN,NaN,NaN,NaN,5,0,8,0,0,2800.0,NaN,NaN,unknown,NaN,unknown
3,6862950349,KUMARA - SUNSCREEN Wardah | Madame Gie Madame Protect Me Sunscreen SPF 30 - SkinCare,4.79,567,31000.0,26500.0,402,229,KUMARA STORE KOSMETIK,4.79,741,92,1.0,4.000000,61100,2700,NaN,0,71550000.0,24 Bulan,402.0,Java,150g,others,NaN,NaN,unknown,others,NaN,NaN,NaN,NaN,4,0,6,0,0,4500.0,24.0,150.0,medium,NaN,unknown
4,22877274329,Madame Gie Madame Protect Me Sunscreen Spray Mist SPF 50 PA++++ | Isi 50ml & 100ml,4.86,35,34000.0,29000.0,9,50,cherrishopie,4.82,460,81,1.0,8.000000,9600,138,NaN,0,4002000.0,24 Bulan,9.0,Unknown,NaN,spray,NaN,NaN,unknown,others,NaN,NaN,NaN,NaN,2,0,7,0,0,5000.0,24.0,NaN,unknown,NaN,unknown


### Drop unnecessary columns

In [56]:
cols = ["Masa Penyimpanan", "Berat Produk", "No. Izin Edar (BPOM, lainnya)", "Tanggal Kedaluwarsa", "Jenis Kulit", "Jumlah Produk Dalam Kemasan", "Ukuran Per Produk", "Stok"]

df = df.drop(
        columns=[col for col in cols if col in df.columns],
        errors="ignore"
    )

### Clean Brand

In [57]:
# Fill NaN first
df['brand'] = df['brand'].fillna("Unknown")

# Get value counts
brand_counts = df['brand'].value_counts()

# Define threshold, eg. 0.1% of total rows
threshold = len(df) * 0.001

# Create a new column: keep brand if count >= threshold, else "Other"
df['brand'] = df['brand'].apply(lambda x: x if brand_counts.get(x, 0) >= threshold else "Other")

df.brand.value_counts()

brand
Unknown           5650
Other              235
Azarine             30
Skintific           25
Anessa              21
WARDAH              15
Pigeon              13
Barbie Beauty       13
KAHF                13
SKIN AQUA           11
Grace and Glow      11
EMINA               10
Originote           10
Bioaqua              9
MADAME GIE           8
SCI Beauty           7
SCARLETT             7
Name: count, dtype: int64

### Rename column names to English

In [58]:
df = df.rename(columns={
    "Dikirim Dari": "shipped_from",
    "Formulasi": "formulation",
    "SPF": "spf",
    "Negara Asal": "country_of_origin",
    "Volume": "volume",
    "berat_gram": "weight_gram",
    "berat_group": "weight_group"
})

### Final Dataset Preview

In [59]:
df.head()

,id,title,rating,reviews,initial_price,final_price,stock,favorite,seller_name,seller_rating,seller_products,seller_chats_responded_percentage,seller_chat_time_reply,seller_joined_date,seller_followers,sold,brand,flash_sale,gmv_cal,shipped_from,formulation,spf,country_of_origin,volume,variations_count,voucher_status,image_count,video_count,review,discount,storage_duration_months,weight_gram,weight_group,volume_ml,volume_group
0,74170424,Skin Aqua UV Moisture Milk SPF 50 | Moisture Gel SPF 30 | Mild Milk SPF 25 | Whitening Milk SPF 50 | Tone Up UV Essence Lavender | Mint Green SPF 50+ | Sunscreen Mist,4.92,16038,49900.0,41300.0,1580,10,megabeautyfashindo,4.91,784,99,1.0,7.000000,114400,10,Unknown,0,413000.0,Jabodetabek,others,unknown,others,NaN,7,0,8,0,0,8600.0,24.0,NaN,unknown,NaN,unknown
1,19980336891,Skin Aqua UV Whitening Milk SPF 50 PA++++ Sunscreen,4.78,112,61000.0,42500.0,50,128,DesakiCosmetik,4.87,39,92,1.0,0.333333,4200,468,Unknown,0,19890000.0,Jabodetabek,others,unknown,others,NaN,0,0,6,0,0,18500.0,12.0,NaN,unknown,NaN,unknown
2,18865050074,AZARINE BT21 BTS Body Guard Moisturizer Sunscreen Serum SPF 50 PA++++ | Magical Luv Sweet Treats Baby Sunproof Monster Sun O Clock Young and Free,5.00,18,65000.0,62200.0,16,5,Made By Caramel,4.98,414,100,1.0,5.000000,4400,40,Unknown,0,2488000.0,Sumatra,others,unknown,others,NaN,5,0,8,0,0,2800.0,NaN,NaN,unknown,NaN,unknown
3,6862950349,KUMARA - SUNSCREEN Wardah | Madame Gie Madame Protect Me Sunscreen SPF 30 - SkinCare,4.79,567,31000.0,26500.0,402,229,KUMARA STORE KOSMETIK,4.79,741,92,1.0,4.000000,61100,2700,Unknown,0,71550000.0,Java,others,unknown,others,NaN,4,0,6,0,0,4500.0,24.0,150.0,medium,NaN,unknown
4,22877274329,Madame Gie Madame Protect Me Sunscreen Spray Mist SPF 50 PA++++ | Isi 50ml & 100ml,4.86,35,34000.0,29000.0,9,50,cherrishopie,4.82,460,81,1.0,8.000000,9600,138,Unknown,0,4002000.0,Unknown,spray,unknown,others,NaN,2,0,7,0,0,5000.0,24.0,NaN,unknown,NaN,unknown


In [ ]:
## Save the cleaned dataset to a new CSV
df.to_csv("C:\\Users\\Acer\\Downloads\\COMP4501-FYP\\cleaned_dataset_final_2\\sunscreen_final_cleaned.csv", index=False)